In [1]:
!pip install rectools[lightfm] matplotlib seaborn polars


[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
# from typing import List
# from tqdm import tqdm
# import random
# from scipy.sparse import csr_matrix
# from implicit.als import AlternatingLeastSquares

In [3]:
from copy import deepcopy
from itertools import combinations
import pickle
import typing as tp
 
from rectools.models import LightFMWrapperModel 
from rectools.dataset import Dataset
# from rectools.dataset import Dataset as LFMDataset
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix
from sklearn.preprocessing import normalize
# from transliterate import translit
from lightfm import LightFM
# from lightfm.data import Dataset



c:\Users\kanze\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\kanze\AppData\Local\Programs\Python\Python310\lib\site-packages\lightfm\_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


In [4]:
df_users = pd.read_csv('data_original/data_original/users.csv')
df_interactions = pd.read_csv('data_original/data_original/interactions.csv')
df_items = pd.read_csv('data_original/data_original/items.csv')

In [5]:
df_users = df_users.drop(df_users.index[-1])
df_interactions = df_interactions.drop(df_interactions.index[-1])
df_items = df_items.drop(df_items.index[-1])


In [6]:
df_interactions.info

<bound method DataFrame.info of          user_id  item_id last_watch_dt  total_dur  watched_pct
0         176549     9506    2021-05-11       4250         72.0
1         699317     1659    2021-05-29       8317        100.0
2         656683     7107    2021-05-09         10          0.0
3         864613     7638    2021-07-05      14483        100.0
4         964868     9506    2021-04-30       6725        100.0
...          ...      ...           ...        ...          ...
5476245   786732     4880    2021-05-12        753          0.0
5476246   648596    12225    2021-08-13         76          0.0
5476247   546862     9673    2021-04-13       2308         49.0
5476248   697262    15297    2021-08-20      18307         63.0
5476249   384202    16197    2021-04-19       6203        100.0

[5476250 rows x 5 columns]>

In [7]:
df_interactions = pd.read_csv('data_original/data_original/interactions.csv')
df_interactions['last_watch_dt'] = pd.to_datetime(df_interactions['last_watch_dt'], errors='coerce')

In [8]:
# Добавление нового столбца
df_interactions['completed'] = (df_interactions['watched_pct'] >= 60).astype(int)

Набор на обучение


In [9]:
test_size_days=10

In [10]:
from datetime import datetime, timedelta

# Тестовый промежуток времени равен 10 дней
max_date = df_interactions['last_watch_dt'].max()
test_start = max_date - timedelta(days=test_size_days)

In [11]:
# Разделение на тренировочный и тестовый
df_interactions_train = df_interactions[df_interactions['last_watch_dt'] < test_start]
df_interactions_test = df_interactions[df_interactions['last_watch_dt'] >= test_start]

In [12]:
# не обрабатываем холодных пользователей
df_interactions_test = df_interactions_test.loc[df_interactions_test["user_id"].isin(df_interactions_train["user_id"]) & df_interactions_test["item_id"].isin(df_interactions_train["item_id"])]

In [13]:
df_interactions_train = pd.merge(df_interactions_train, df_items[["item_id", "title", "genres"]], on="item_id")
df_interactions_test = pd.merge(df_interactions_test, df_items[["item_id", "title", "genres"]], on="item_id")

In [14]:
train_users = df_interactions_train['user_id'].unique()
test_users = df_interactions_test['user_id'].unique()
all_items = df_interactions['item_id'].unique().tolist()

In [15]:
random_user = df_interactions_test.sample(1)  
random_user

,user_id,item_id,last_watch_dt,total_dur,watched_pct,completed,title,genres
382418,730347,13424,2021-08-16,740,14.0,0,Другой мир: Войны крови,"драмы, фантастика, военные, боевики, ужасы, фэ..."


In [16]:
random_user_id = random_user["user_id"].values[0]

In [17]:
watched_items = df_interactions_train[df_interactions_train["user_id"] == random_user_id]["item_id"].unique()
watched_items


array([ 6067, 12995,  9857,  9728,  2924, 15379, 15997,  6440,  3734,
        3071, 10440, 10489,  4880,  7107, 13706,  9996,  8283,  4151,
       11237, 12259,  4106, 12324, 12760,  7793,  9081], dtype=int64)

In [18]:
df_interactions_train = pd.concat([df_interactions_train, random_user], sort=False)

In [19]:
import rectools
print(dir(rectools.models))

['BERT4RecModel', 'DSSMModel', 'EASEModel', 'HSTUModel', 'ImplicitALSWrapperModel', 'ImplicitBPRWrapperModel', 'ImplicitItemKNNWrapperModel', 'LightFMWrapperModel', 'PopularInCategoryModel', 'PopularModel', 'PureSVDModel', 'RandomModel', 'SASRecModel', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'base', 'ease', 'implicit_als', 'implicit_bpr', 'implicit_knn', 'lightfm', 'load_model', 'model_from_config', 'model_from_params', 'nn', 'popular', 'popular_in_category', 'pure_svd', 'random', 'rank', 'serialization', 'utils', 'vector']


In [20]:
import lightfm
print(dir(lightfm))

['LightFM', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', '_lightfm_fast', '_lightfm_fast_no_openmp', 'lightfm', 'version']


In [21]:
# from rectools.dataset import Dataset

# dataset = Dataset.construct(
#     interactions_df=df_interactions_train[['user_id', 'item_id', 'completed']]
# )

In [22]:
df_interactions_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4811276 entries, 0 to 382418
Data columns (total 8 columns):
 #   Column         Dtype         
---  ------         -----         
 0   user_id        int64         
 1   item_id        int64         
 2   last_watch_dt  datetime64[ns]
 3   total_dur      int64         
 4   watched_pct    float64       
 5   completed      int32         
 6   title          object        
 7   genres         object        
dtypes: datetime64[ns](1), float64(1), int32(1), int64(3), object(2)
memory usage: 312.0+ MB


In [23]:
from rectools import Columns

In [24]:
# Проверим наличие пропусков
print(df_interactions_train.isna().sum())

# Удалим строки, где есть NaN в ключевых колонках
df_interactions_train = df_interactions_train.dropna(subset=["user_id", "item_id", "watched_pct", "completed"], how="any")

# Удалим строки, где watched_pct = inf или -inf
df_interactions_train = df_interactions_train[np.isfinite(df_interactions["watched_pct"])]

# Проверим снова
print(df_interactions_train.isna().sum())

user_id            0
item_id            0
last_watch_dt      0
total_dur          0
watched_pct      826
completed          0
title              0
genres             0
dtype: int64


C:\Users\kanze\AppData\Local\Temp\ipykernel_8252\1842086321.py:8: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_interactions_train = df_interactions_train[np.isfinite(df_interactions["watched_pct"])]


user_id          0
item_id          0
last_watch_dt    0
total_dur        0
watched_pct      0
completed        0
title            0
genres           0
dtype: int64


In [25]:
interactions_df = df_interactions_train.copy()

interactions_df.dropna(subset=['user_id','item_id', 'last_watch_dt', 'watched_pct', 'total_dur','completed'])

interactions_df.reset_index()

interactions_df.drop(interactions_df.tail(2).index, inplace=True)

In [26]:
ratings = pd.DataFrame()
ratings[Columns.User] = interactions_df['user_id']
ratings[Columns.Item] = interactions_df['item_id']
ratings[Columns.Weight] = interactions_df['watched_pct']
ratings[Columns.Datetime] = interactions_df['last_watch_dt']
# ratings['completed'] = df_interactions_train['completed']
# ratings['title'] = df_interactions_train['title']
# ratings['genres'] = df_interactions_train['genres']

In [27]:
# Посмотрим какие параметры принимает construct
import inspect
print(inspect.signature(Dataset.construct))

# Или посмотрим документацию
help(Dataset.construct)

(interactions_df: pandas.core.frame.DataFrame, user_features_df: Optional[pandas.core.frame.DataFrame] = None, cat_user_features: Iterable[str] = (), make_dense_user_features: bool = False, item_features_df: Optional[pandas.core.frame.DataFrame] = None, cat_item_features: Iterable[str] = (), make_dense_item_features: bool = False, keep_extra_cols: bool = False) -> 'Dataset'
Help on method construct in module rectools.dataset.dataset:

construct(interactions_df: pandas.core.frame.DataFrame, user_features_df: Optional[pandas.core.frame.DataFrame] = None, cat_user_features: Iterable[str] = (), make_dense_user_features: bool = False, item_features_df: Optional[pandas.core.frame.DataFrame] = None, cat_item_features: Iterable[str] = (), make_dense_item_features: bool = False, keep_extra_cols: bool = False) -> 'Dataset' method of builtins.type instance
    Class method for convenient `Dataset` creation.
    
    Use it to create dataset from raw data.
    
    Parameters
    ----------
    in

In [28]:
from rectools import  dataset
# Создаем Dataset из rectools
dataset = dataset.Dataset.construct(ratings)

users_uniq = ratings[Columns.User].unique()



In [29]:
# Создаем и обучаем модель
model = LightFMWrapperModel(
    model=LightFM(no_components=30, learning_rate=0.05, loss="bpr", random_state=42),
    epochs=100
)

In [ ]:
dataset

Dataset(user_id_map=IdMap(external_ids=array([176549, 699317, 656683, ..., 911142, 882138, 805174], dtype=int64)), item_id_map=IdMap(external_ids=array([ 9506,  1659,  7107, ..., 13516, 13019, 10542], dtype=int64)), interactions=Interactions(df=         user_id  item_id  weight   datetime
0              0        0    72.0 2021-05-11
1              1        1   100.0 2021-05-29
2              2        2     0.0 2021-05-09
3              3        3   100.0 2021-07-05
4              4        0   100.0 2021-04-30
...          ...      ...     ...        ...
4809723   180313       58    98.0 2021-04-21
4809724   166488     2270     1.0 2021-05-29
4809725    67332      214   100.0 2021-08-02
4809726    38719      128     0.0 2021-05-12
4809727   200228     2485    49.0 2021-04-13

[4809728 rows x 4 columns]), user_features=None, item_features=None)

: 

In [ ]:
model.fit(dataset)

In [ ]:
# from lightfm.data import Dataset

# dataset = Dataset(ratings)
# user_uniq = ratings['user_id'].unique

In [ ]:
# dataset = Dataset()
# dataset.fit(users=df_interactions_train['user_id'].unique(), 
#             items=df_interactions_train['item_id'].unique())



# 3. Теперь обучаем модель на матрице взаимодействий
# model = LightFM(no_components=30, learning_rate=0.05, loss="bpr", random_state=42)
# model.fit(interactions, epochs=100)

In [ ]:
# # 2. Строим матрицу взаимодействий
# interactions, weights = dataset.build_interactions(
#     [(row['user_id'], row['item_id'], row['watched_pct']) 
#      for idx, row in df_interactions_train.iterrows()]
# )

In [ ]:
# recommendations = model.recommend(
#     users=[random_user_id],
#     dataset=dataset,
#     k=10,
#     filter_viewed=True
# )